# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# Securely fetch the HF token
try:
    HF_TOKEN = userdata.get('HF_TOKEN')

    # The dataset requires a config name. We use 'fact_content_query_90d' for GSC data.
    ds = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d", token=HF_TOKEN)
    df = ds['train'].to_pandas()

    # This table uses 'window_end' as the primary time marker
    if 'window_end' in df.columns:
        df['date'] = pd.to_datetime(df['window_end'])
        # Filtering for a specific window in early 2026
        df_mid = df[df['date'].dt.strftime('%Y-%m') == '2026-03'].copy()

        if df_mid.empty:
            # If March is empty, let's take the first available month to ensure df_mid exists
            available_months = df['date'].dt.strftime('%Y-%m').unique()
            print(f"Warning: 2026-03 not found. Available months: {available_months[:3]}")
            df_mid = df[df['date'].dt.strftime('%Y-%m') == available_months[0]].copy()

        print(f"Success! Loaded {len(df_mid)} rows for analysis.")
        display(df_mid.head(2))
    else:
        print(f"Error: Expected time columns not found. Columns: {df.columns.tolist()}")
except Exception as e:
    print(f"Error loading data: {e}")

Success! Loaded 2414248 rows for analysis.


,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share,date
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102,2026-06-30
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102,2026-06-30


## 1. Unit of analysis + time window

**The Contract (5 Answers):**
1. **One row means:** A unique combination of a `client`, `content` (page), and `query` within a 90-day aggregation window ending on `window_end`.
2. **Tables used:** `fact_content_query_90d` (GSC-derived warehouse table).
3. **Time window:** Mid-panel month for development is June 2026 (based on available data); future prediction targets the following 30-day window.
4. **Label/Proxy:** `label_high_performance` (Binary: Is `clicks_last30` above the median?).
5. **Excluded:** Queries with zero impressions in the last 90 days are excluded because the warehouse grain requires an active impression to exist.

### 1) The Contract - Plain Words
1. **One Row Means:** A unique combination of `query` + `landing_page` for a specific `date`.
2. **Tables Used:** Search Console data (joined with Analytics if available in the slice).
3. **Time Window:** Mid-panel training on March 2026; Evaluation on April 2026.
4. **Label/Proxy:** `is_high_click` (Boolean: clicks > median) - we want to rank queries that drive traffic.
5. **Excluded:** Brand-name queries (e.g., "FlyRank") to focus on discovery keywords.

In [6]:
# Prove three facts with three queries on a mid-panel month

# 1. The Grain: Confirming hash IDs + date define a unique row
dup_keys = ['client_hash_id', 'content_hash_id', 'query_hash_id', 'window_end']
duplicates = df_mid.duplicated(subset=dup_keys).sum()
print(f"Fact 1 (Grain): Duplicate rows = {duplicates}")

# 2. Slice Count and Date Span
print(f"Fact 2 (Stats): {len(df_mid):,} rows found between {df_mid['date'].min()} and {df_mid['date'].max()}")

# 3. Availability: Filter with IS TRUE (or Boolean condition) for impression data
availability_check = (df_mid['impressions_90d'] > 0).all()
print(f"Fact 3 (Availability): Impressions > 0 is {availability_check} for all rows.")

In [12]:
# 1. Verify Grain: (client_hash_id, content_hash_id, query_hash_id, window_end) should be unique
duplicate_keys = ['client_hash_id', 'content_hash_id', 'query_hash_id', 'window_end']
duplicates = df_mid.duplicated(subset=duplicate_keys).sum()
print(f"Duplicate rows based on ID keys: {duplicates}")

# 2. Slice row count and date span
print(f"Row count: {len(df_mid)}")
print(f"Date span (window_end): {df_mid['date'].min()} to {df_mid['date'].max()}")

# 3. Availability Check: How many have valid 90d impression data?
valid_rows = (df_mid['impressions_90d'] > 0).sum()
print(f"Rows with impressions_90d > 0: {valid_rows} ({(valid_rows/len(df_mid)):.2%})")

Duplicate rows based on ID keys: 0
Row count: 2414248
Date span (window_end): 2026-06-30 00:00:00 to 2026-06-30 00:00:00
Rows with impressions_90d > 0: 2414248 (100.00%)


## 2. Fields: feature / label / context / excluded

- **Features:** `query_token_count` (complexity), `avg_position_90d` (visibility), `rare_query_count` (specificity).
- **Label:** `label_high_performance` (derived from `clicks_last30`).
- **Context:** `window_end` (temporal anchor), `client_hash_id` (segmentation).
- **Excluded:** `trap_leakage` (we exclude the total 90d clicks during modeling because they contain the outcome we want to predict).

In [ ]:
# Build the small feature frame (5 features max)
features = df_mid.copy()

# 1. feat_token_count: Knowable at decision moment because it is derived from the query string.
features['feat_token_count'] = features['query_token_count']

# 2. feat_avg_pos: Knowable because it represents the historical rank up to the decision point.
features['feat_avg_pos'] = features['avg_position_90d']

# 3. feat_rare_flag: Knowable because the warehouse flags rare queries during the sync.
features['feat_is_rare'] = (features['rare_query_count'] > 0).astype(int)

# 4. feat_visibility: Knowable at decision moment as it uses 90-day historical window totals.
features['feat_visibility'] = features['impressions_90d'] / features['content_total_impressions_90d'].replace(0, 1)

# 5. feat_char_len: Knowable because the query text is provided at the time of search.
features['feat_char_len'] = features['query_char_count']

# THE TRAP: Adding ONE label-derived column on purpose
features['trap_leakage'] = features['clicks_90d']

# Create label
features['label_high_performance'] = (features['clicks_last30'] > features['clicks_last30'].median()).astype(int)

print("Feature frame built. Showing trap:")
display(features[['feat_token_count', 'trap_leakage', 'label_high_performance']].head())

# Delete the trap to keep the honest number
features.drop(columns=['trap_leakage'], inplace=True)
print("\nLeakage trap removed.")

In [13]:
# Building 5 Features from the fact_content_query_90d table
features = df_mid.copy()

# 1. query_complexity: Knowable from query_token_count
features['feat_token_count'] = features['query_token_count']

# 2. recent_click_through: Knowable at decision moment (lagged 30d vs 90d window)
features['feat_clicks_last30'] = features['clicks_last30']

# 3. relative_position: Knowable from warehouse position data
features['feat_avg_pos'] = features['avg_position_90d']

# 4. content_dominance: Ratio of this query to total content impressions
features['feat_visibility_share'] = features['impressions_90d'] / features['content_total_impressions_90d'].replace(0, 1)

# 5. rare_query_flag: Binary feature from warehouse
features['feat_is_rare'] = (features['rare_query_count'] > 0).astype(int)

# THE TRAP: Adding 'clicks_90d' which contains the outcome we are predicting
features['trap_leakage'] = features['clicks_90d']

# Target label: High performance in the last 30 days
features['label_high_performance'] = (features['clicks_last30'] > features['clicks_last30'].median()).astype(int)

display(features[['query_hash_id', 'feat_token_count', 'feat_is_rare', 'trap_leakage', 'label_high_performance']].head())

# REMOVE THE TRAP
features.drop(columns=['trap_leakage'], inplace=True)
print("Trap removed. Dataset is now leakage-free.")

,query_hash_id,feat_token_count,feat_is_rare,trap_leakage,label_high_performance
0,query_58b1b001f839d699,3,1,0,0
1,query_922b8eca2a24cd34,7,1,0,0
2,query_9f0c36a6ae2a6a99,2,1,0,0
3,query_a032820b5467e996,4,1,0,0
4,query_ba1a2f131961c5da,3,1,0,0


Trap removed. Dataset is now leakage-free.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

**Named Limitation:** **Selection Bias.** This slice only tracks queries that actually resulted in an impression. It cannot tell us about "lost" queries that are relevant to our content but never achieved a high enough rank to be recorded by Search Console.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Limitation
**Zero-Exposure queries:** This slice only includes queries that generated at least one impression. We cannot model 'lost potential' for queries that never appeared in the top 100 results during this window.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.